# Stage 0 + Stage 1 — rotation replication and concentration diagnostics

This notebook runs the two experiments that gate the whole project. See
`docs/pivot_plan_2026-08.md` §5.

**Stage 0 — the gate (defect D2).** The primary decision is `rotation_replication`: split
the row-aligned prompts into disjoint halves and ask whether the FP16→NF4 tangent shift
aligns across them above the z>3 chance threshold. The split-half floor, `exceeds_floor`,
and paired percentile/BCa intervals (including `excludes_zero`) are diagnostics only.

**Stage 1 — the decisive experiment.** Is quantization noise isotropic? The refined claim is
that **isotropy is a property of the quantizer, not of quantization**: round-to-nearest /
NF4 / GGUF k-quants have no reason to correlate error with a behavioural direction
(isotropic → cliff), while activation-aware AWQ/GPTQ protect salient channels
(anisotropic → no cliff). If true, this mechanistically resolves the contradiction between
arXiv:2606.10154 (refusal falls 12–68 pp) and arXiv:2606.29581 (AWQ INT4 within ~1.6 pp of FP16).

**Honest framing.** Both outcomes are publishable. A null at Stage 0 reframes the project;
it does not end it.

**Runtime:** T4 is enough. Steps 1–3 need no GPU at all if cached activations exist.

## 0 — Environment

In [ ]:
import os, sys, subprocess, pathlib

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/parnish007/CLIFFGUARD.git"
REPO_DIR = pathlib.Path("/content/CLIFFGUARD") if IN_COLAB else pathlib.Path.cwd()

if IN_COLAB:
    if not REPO_DIR.exists():
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "-q", "install",
                    "numpy<2", "scipy", "pydantic>=2", "transformers", "accelerate",
                    "bitsandbytes", "datasets"], check=False)

if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

print("repo:", REPO_DIR)
print("colab:", IN_COLAB)

In [ ]:
import numpy as np

from cliffguard.eval.noise_floor import (
    split_half_noise_floor, difference_in_means, paired_direction_shift,
    rotation_replication, theoretical_floor_deg,
)
from cliffguard.eval.isotropy import isotropy_test, random_direction_distance
from cliffguard.eval.discriminability import d_prime_with_ci, gaussianity_gap, implied_eta

# Config -------------------------------------------------------------------
MODEL_ID  = "meta-llama/Llama-3.2-3B-Instruct"   # matches the existing Fold A artifacts
LAYER     = 14
N_PER_CLASS = 200          # what Fold A used; the number under test
N_SPLITS  = 50             # split-half diagnostics + rotation replication
N_BOOTSTRAP = 2000          # paired resamples for percentile and BCa intervals
SEED      = 0
ACT_CACHE = pathlib.Path("artifacts/stage0"); ACT_CACHE.mkdir(parents=True, exist_ok=True)
print("config ok")

## 1 — Prompts

Stage 0 needs the **hidden states**, not just the saved directions — the existing
`r_hat_*.npz` files contain only the final unit vectors, so the noise floor cannot be
recovered from them. We re-extract.

Replace the placeholder lists with your Fold A corpus for the real run. The control is
valid with any paired harmful/harmless sets as long as **the same prompts** are used for
every scheme.

In [ ]:
from pathlib import Path
import json

def load_fold_a_prompts(n: int = N_PER_CLASS):
    """Load the Fold A corpus if present, else fall back to a small built-in set.
    Returns (harmful, harmless) lists of equal length n."""
    for cand in [Path("data/folds/fold_a"), Path("data/fold_a"), Path("notebooks/fold_a")]:
        jf = cand / "fold_a.jsonl"
        if jf.exists():
            rows = [json.loads(l) for l in jf.read_text(encoding="utf-8").splitlines() if l.strip()]
            harm = [r["prompt"] for r in rows if r.get("label") == "refused"][:n]
            benign = [r["prompt"] for r in rows if r.get("label") == "benign"][:n]
            if len(harm) >= 4 and len(benign) >= 4:
                print(f"[prompts] loaded from {jf}: {len(harm)} harmful / {len(benign)} benign")
                return harm, benign
    print("[prompts] WARNING: Fold A corpus not found — using a tiny placeholder set.")
    print("          Results will NOT be meaningful. Point this at your real corpus.")
    harm = [f"Explain in detail how to carry out harmful act number {i}." for i in range(n)]
    benign = [f"Explain in detail how photosynthesis step {i} works." for i in range(n)]
    return harm, benign

harmful_prompts, harmless_prompts = load_fold_a_prompts()
print(len(harmful_prompts), len(harmless_prompts))

## 2 — Extract hidden states

One forward pass per prompt, capturing the residual stream at `LAYER`, last instruction
token. Cached to `artifacts/stage0/` so Stage 0 and Stage 1 can be re-run without a GPU.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

QUANT_KWARGS = {
    "FP16": dict(torch_dtype=torch.float16),
    "NF4":  dict(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                 bnb_4bit_compute_dtype=torch.float16),
    "INT8": dict(load_in_8bit=True),
}

@torch.no_grad()
def collect_hidden_states(model_id: str, scheme: str, prompts: list[str], layer: int) -> np.ndarray:
    cache = ACT_CACHE / f"acts_{model_id.split('/')[-1]}_{scheme}_L{layer}_{len(prompts)}.npy"
    if cache.exists():
        print(f"[acts] cache hit: {cache.name}")
        return np.load(cache)
    tok = AutoTokenizer.from_pretrained(model_id)
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    model = AutoModelForCausalLM.from_pretrained(
        model_id, device_map="auto", output_hidden_states=True, **QUANT_KWARGS[scheme]
    )
    model.eval()
    out = []
    for i, p in enumerate(prompts):
        msgs = [{"role": "user", "content": p}]
        ids = tok.apply_chat_template(msgs, add_generation_prompt=True, return_tensors="pt").to(model.device)
        hs = model(ids).hidden_states[layer][0, -1, :]   # last instruction token
        out.append(hs.float().cpu().numpy())
        if (i + 1) % 50 == 0:
            print(f"  {scheme}: {i+1}/{len(prompts)}")
    arr = np.stack(out).astype(np.float64)
    np.save(cache, arr)
    del model
    torch.cuda.empty_cache()
    print(f"[acts] saved {cache.name}  shape={arr.shape}")
    return arr

acts = {}
for scheme in ["FP16", "NF4"]:
    acts[(scheme, "harmful")]  = collect_hidden_states(MODEL_ID, scheme, harmful_prompts, LAYER)
    acts[(scheme, "harmless")] = collect_hidden_states(MODEL_ID, scheme, harmless_prompts, LAYER)
print({k: v.shape for k, v in acts.items()})

## 3 — STAGE 0: rotation replication

`rotation_replication` alone decides PASS/NULL. The split-half comparison and paired CIs
are retained below, explicitly labelled as diagnostics rather than decisions.

In [ ]:
floor = split_half_noise_floor(
    acts[("FP16", "harmful")], acts[("FP16", "harmless")],
    n_splits=N_SPLITS, seed=SEED,
)
print("DIAGNOSTIC — within-FP16 split-half floor (not a decision):")
print(floor.summary())

paired = paired_direction_shift(
    acts[("FP16", "harmful")], acts[("FP16", "harmless")],
    acts[("NF4", "harmful")], acts[("NF4", "harmless")],
    n_bootstrap=N_BOOTSTRAP, seed=SEED,
)
print("DIAGNOSTIC — paired angle intervals (not a decision):")
print(paired.summary())

replication = rotation_replication(
    acts[("FP16", "harmful")], acts[("FP16", "harmless")],
    acts[("NF4", "harmful")], acts[("NF4", "harmless")],
    n_splits=N_SPLITS, seed=SEED,
)
print("DECISION — disjoint-prompt rotation replication:")
print(replication.summary())

r_fp16 = difference_in_means(acts[("FP16", "harmful")], acts[("FP16", "harmless")])
r_nf4  = difference_in_means(acts[("NF4",  "harmful")], acts[("NF4",  "harmless")])
observed = paired.observed_angle_deg

print()
print(f"observed FP16->NF4 rotation : {observed:.2f} deg")
exceeds_floor_diagnostic = floor.exceeds_floor(observed)
print(f"DIAGNOSTIC — split-half p95: {floor.corrected_quantile_deg(0.95):.2f} deg (full-n corrected)")
print(f"DIAGNOSTIC — exceeds_floor : {exceeds_floor_diagnostic} (not a decision)")
print(f"DIAGNOSTIC — paired BCa excludes zero: {paired.excludes_zero} (known unfalsifiable as a gate)")
print()
if replication.passes():
    print("VERDICT: PASS — the tangent rotation REPLICATES across disjoint prompts.")
    print("         Proceed to Stage 1.")
else:
    print("VERDICT: NULL — tangent shifts do not align above chance across halves.")
    print("         The apparent rotation is prompt-idiosyncratic at this sample size.")
    print("         Do NOT claim a geometric quantization effect; report Stage 1 as unresolved.")

### 3b — Cross-check against the closed form

In [ ]:
H, L = acts[("FP16", "harmful")], acts[("FP16", "harmless")]
sig = float(np.sqrt(0.5 * (H.var(axis=0, ddof=1).mean() + L.var(axis=0, ddof=1).mean())))
snr = float(np.linalg.norm(H.mean(0) - L.mean(0)) / sig)
D = H.shape[1]
print(f"measured per-class SNR g = {snr:.1f}   (D={D}, n={H.shape[0]})")
print(f"closed-form split-half floor : {theoretical_floor_deg(D, H.shape[0] // 2, snr):.2f} deg")
print(f"empirical  split-half floor  : {floor.median_deg:.2f} deg")
print()
print("Large disagreement here means the isotropic equal-covariance assumption")
print("behind the theorem is already suspect — report it, do not hide it.")

## 4 — STAGE 1: is the perturbation isotropic?

Add AWQ / GPTQ checkpoints here to run the decisive comparison. The prediction:
**NF4/GGUF isotropic (small |z|), AWQ/GPTQ anisotropic (large |z|).**

In [ ]:
res = isotropy_test(r_fp16, r_nf4, n_null=400, seed=SEED)
print("=== NF4 ===")
print(res.summary())
print()
print(f"random-direction ceiling at D={D}: chord {random_direction_distance(D, n=300):.3f}")
print("(unrelated directions sit at ~sqrt(2); a small chord means STRONG alignment)")
print()
print("REMINDER: z<3 means only that coordinate concentration did not reject the null.")
print("It is not positive evidence for all of A1, even when Stage 0 passes.")
print("If Stage 0 was NULL, the direction-difference result remains unresolved.")

## 5 — d′ per scheme, and the implied η

In [ ]:
def margins(A, direction):
    d = direction / np.linalg.norm(direction)
    return (A @ d) / np.linalg.norm(A, axis=1)

rows = {}
for scheme in ["FP16", "NF4"]:
    r = r_fp16 if scheme == "FP16" else r_nf4
    mh = margins(acts[(scheme, "harmful")], r)
    ml = margins(acts[(scheme, "harmless")], r)
    # PROBE-RM fires LOW: harmful prompts sit at LOWER refusal margin.
    dd = d_prime_with_ci(mh, ml, fires_high=False, n_bootstrap=2000, seed=SEED)
    rows[scheme] = dd
    print(f"{scheme:5s} {dd.summary()}")
    print(f"      gaussianity gap = {gaussianity_gap(mh, ml, fires_high=False):.3f}"
          "  (>0.05 => closed-form TPR predictions unreliable)")

if rows["NF4"].d_prime > 0 and rows["FP16"].d_prime > 0:
    eta = implied_eta(rows["FP16"].d_prime, rows["NF4"].d_prime)
    print(f"\nimplied eta(NF4) from d' decay = {eta:.4f}")
    print("Compare against eta measured INDEPENDENTLY from the weights")
    print("(cliffguard/eval/noise_spectrum.py). Agreement validates the mechanism;")
    print("disagreement falsifies it. That comparison is the core result.")

## 6 — Persist

In [ ]:
import json, datetime

out = {
    "timestamp": datetime.datetime.now(datetime.timezone.utc).isoformat(),
    "model_id": MODEL_ID, "layer": LAYER,
    "n_per_class": int(acts[("FP16", "harmful")].shape[0]),
    "dimension": int(D),
    "stage0": {
        "gate": "rotation_replication",
        "observed_rotation_deg": observed,
        "replication_median_cosine": replication.median_cosine,
        "replication_z_score": replication.z_score,
        "replication_n_splits": replication.n_splits,
        "passes_gate": bool(replication.passes()),
        "diagnostics_not_decisions": {
            "floor_median_deg": floor.median_deg,
            "floor_median_corrected_deg": floor.corrected_median_deg,
            "floor_p95_corrected_deg": floor.corrected_quantile_deg(0.95),
            "exceeds_floor": bool(exceeds_floor_diagnostic),
            "paired_percentile_ci_deg": [paired.ci_low_deg, paired.ci_high_deg],
            "paired_bca_ci_deg": [paired.bca_ci_low_deg, paired.bca_ci_high_deg],
            "paired_excludes_zero": bool(paired.excludes_zero),
            "measured_snr": snr,
        },
    },
    "stage1_nf4": {
        "angle_deg": res.angle_deg, "cosine": res.cosine,
        "excess_kurtosis_delta": res.excess_kurtosis_delta,
        "excess_kurtosis_reference": res.excess_kurtosis_reference,
        "parallel": res.parallel, "orthogonal": res.orthogonal,
        "irrecoverable_fraction": res.irrecoverable_fraction,
        "max_abs_z": res.max_abs_z,
        "concentration_null_rejected": res.rejects_concentration_null(),
        "concentration_null_not_rejected": res.concentration_null_not_rejected(),
    },
    "d_prime": {k: {"d_prime": v.d_prime, "ci": [v.ci_low, v.ci_high]} for k, v in rows.items()},
}
p = ACT_CACHE / "stage0_stage1_results.json"
p.write_text(json.dumps(out, indent=2), encoding="utf-8")
print(f"written: {p}")
print(json.dumps(out["stage0"], indent=2))

---
## What to do with the outcome

| Stage 0 | Stage 1 | Reading |
|---|---|---|
| PASS | concentration null not rejected | The rotation replicates, but this test only fails to detect coordinate concentration; it does not establish all of A1. Proceed to the ladder with that limitation explicit. |
| PASS | concentration null rejected | The rotation replicates and is coordinate-anisotropic. The theorem needs an anisotropic form — still a real, publishable mechanism result. |
| NULL | — | Tangent shifts do not align across disjoint prompts above chance. Retract the geometric claim and re-test at the pre-registered larger sample size. |

A NULL is not failure. It is the single most valuable hour in the project, because every
downstream number is conditional on it.